# 03 — Sizing closure and launch alternatives

Compare the free sizing pass with the selected battery/propulsion hardware.
AR studies rebuild the sizing inputs. Hand launch remains selected; the bungee
result is a trade requiring separate structure, launch and CAD validation.


## Mass closure and dependency chain

**Inputs:** `design_point.yaml`, `electronics.yaml`, payload YAML and generated
`calibration.yaml`. **Implementation:** `uavdesign/sizing.py` and
`design.inputs_for`. The shared mission supplies air density conditions.

The mass loop is $m_{k+1}=m_{payload}+m_{avionics}+m_{structure}(S_k,b_k)
+m_{propulsion}+m_{battery}+m_{contingency}$. Hand-launch sizing imposes
$V_{release}\geq M_{launch}V_s$, with
$V_s=\sqrt{2mg/(\rho S C_{L,max})}$; span then follows
$b=\sqrt{AR\,S}$. These equations explain why payload, effective $C_{L,max}$
and launch assumptions can grow both structure and wing area.

Plain-text equations: `stall_speed = sqrt(2 * mass * g / (density * area * CLmax))`;
`span = sqrt(aspect_ratio * area)`; `next_mass = sum(component_mass_estimates)`.

The free pass estimates propulsion and battery continuously. The selected pass
uses the discrete BOM masses and pack energy, so achieving mass convergence does
not guarantee the endurance target. The AR trade re-sizes each candidate rather
than stretching one completed CAD model; its results are not fabrication files.

`calibration.yaml: measured` contains **CAD-derived estimates**, not scale
measurements. CAD surface/volume plus assumed density, perimeters and infill
correct the analytic mass model. Agreement between two models sharing those
assumptions is consistency evidence, not physical validation.

**External review:** inspect convergence, mass contingency allocation and payload
interchangeability. Repeat with weighed coupons and hardware; re-check structure,
fit and balance for every candidate retained after the trade.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


In [ ]:
from dataclasses import replace
from uavdesign.sizing import size
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for label, result in (("free", d.free), ("selected", d.selected)):
    axes[0].plot(np.arange(1, len(result.history)+1), result.history, label=label)
axes[0].set(xlabel="Iteration", ylabel="All-up mass [kg]"); axes[0].legend()
axes[1].barh(list(d.selected.masses_g), list(d.selected.masses_g.values()))
axes[1].set(xlabel="Mass [g]")
save(fig, "03_closure_mass.png")
display(pd.DataFrame([{"case": "free", **d.free.summary()}, {"case": "selected", **d.selected.summary()}]))
display(pd.DataFrame([ev.__dict__ for ev in d.evaluations.values()]))
rows = []
base = D.inputs_for(d)
for aspect_ratio in studies["aspect_ratios"]:
    result = size(replace(base, aspect_ratio=aspect_ratio), auw_guess_kg=d.selected.auw_kg)
    rows.append({"aspect_ratio": aspect_ratio, "converged": result.converged,
                 "auw_kg": result.auw_kg, "span_m": result.span_m, "endurance_min": result.endurance_min})
display(pd.DataFrame(rows))
fig, ax = plt.subplots(figsize=(7, 4))
frame = pd.DataFrame(rows)
ax.plot(frame.aspect_ratio, frame.auw_kg, "o-", label="AUW")
ax.set(xlabel="Aspect ratio", ylabel="AUW [kg]")
ax2 = ax.twinx(); ax2.plot(frame.aspect_ratio, frame.endurance_min, "s--", color="C1")
ax2.set_ylabel("Endurance [min]")
save(fig, "03_aspect_ratio_trade.png")


In [ ]:
uncalibrated = D.run(solve_stability=False, calibration={})
display(pd.DataFrame([{"calibration": "analytic only", **uncalibrated.selected.summary()},
                      {"calibration": "loaded CAD feedback", **d.selected.summary()}]))
display(pd.DataFrame(D.launch_trade(d)))
print("Calibration coefficients:", d.calibration)
